# IO II - Problem Set 1

## Part I

### Part I Q3

In [24]:
from scipy.special import logsumexp

x = np.array([0.0, 10.0, 1000.0])

m = np.max(x)
manual = m + np.log(np.sum(np.exp(x - m)))
scipy_val = logsumexp(x)

print("Our function value:", manual)
print("SciPy value:", scipy_val)


Manual stable value: 1000.0
SciPy value: 1000.0


## Part II

In [25]:
import numpy as np
from scipy import integrate
from scipy.stats import norm
from scipy.special import expit
from numpy.polynomial.hermite import hermgauss

X = 0.5
mu = 0.5
sigma = 2.0

### Part II Q1

In [26]:
def binomiallogit(beta, X=X, mu=mu, sigma=sigma):
    return expit(beta * X) * norm.pdf(beta, loc=mu, scale=sigma)


### Part II Q2

In [27]:
true_val, err = integrate.quad(
    binomiallogit,
    -np.inf,
    np.inf,
    epsabs=1e-14,
    epsrel=1e-14,
    limit=200
)

print("True value:", true_val)
print("Estimated integration error:", err)

True value: 0.5514932726366428
Estimated integration error: 6.272478140922507e-15


### Part II Q3

In [28]:

np.random.seed(123)

# 20 draws
beta_draws_20 = np.random.normal(mu, sigma, 20)

logit_values_20 = []
for b in beta_draws_20:
    p = expit(b * X)
    logit_values_20.append(p)

mc_estimate_20 = sum(logit_values_20) / len(logit_values_20)

print("Monte Carlo estimate with 20 draws:", mc_estimate_20)


# 400 draws
beta_draws_400 = np.random.normal(mu, sigma, 400)

logit_values_400 = []
for b in beta_draws_400:
    p = expit(b * X)
    logit_values_400.append(p)

mc_estimate_400 = sum(logit_values_400) / len(logit_values_400)

print("Monte Carlo estimate with 400 draws:", mc_estimate_400)

Monte Carlo estimate with 20 draws: 0.56567973798221
Monte Carlo estimate with 400 draws: 0.5372448174967471


### Part II Q4

In [45]:

def q4_gauss_hermite(k):

    # Hermite nodes and weights
    xs, ws = hermgauss(k)

    beta_values = np.sqrt(2) * sigma * xs + mu

    weights = ws / np.sqrt(np.pi)

    logit_values = []
    for b in beta_values:
        p = expit(b * X)
        logit_values.append(p)

    total = 0
    for i in range(k):
        total += weights[i] * logit_values[i]

    return total, beta_values, weights, logit_values

ans4, beta4, weights4, vals4 = q4_gauss_hermite(4)
ans12, beta12, weights12, vals12 = q4_gauss_hermite(12)

print("k = 4 approximation:", ans4)
print("k = 12 approximation:", ans12)
print("weights sum for k = 4:", sum(weights4))
print("weights sum for k = 12:", sum(weights12))

k = 4 approximation: 0.551313876771185
k = 12 approximation: 0.5514932040654397
weights sum for k = 4: 1.0
weights sum for k = 12: 1.0000000000000002


### Part II Q5

In [30]:
import pandas as pd

results_1d = pd.DataFrame({
    "Method": ["Monte Carlo", "Monte Carlo", "Gauss-Hermite", "Gauss-Hermite"],
    "Points/Draws": [20, 400, 4, 12],
    "Estimate": [mc_estimate_20, mc_estimate_400, ans4, ans12],
    "Error": [mc_estimate_20 - true_val, mc_estimate_400 - true_val, ans4 - true_val, ans12 - true_val],
})

print(results_1d)

          Method  Points/Draws  Estimate         Error
0    Monte Carlo            20  0.565680  1.418647e-02
1    Monte Carlo           400  0.537245 -1.424846e-02
2  Gauss-Hermite             4  0.551314 -1.793959e-04
3  Gauss-Hermite            12  0.551493 -6.857120e-08


### Part II Q6

In [31]:

# New Parameters
mu1 = 0.5
mu2 = 1.0
sigma1 = 2.0
sigma2 = 1.0

X1 = 0.5
X2 = 1.0

def binomial_logit_2d(beta1, beta2, X1=X1, X2=X2):
    return expit(beta1 * X1 + beta2 * X2)

#True Value
def integrand_2d(beta2, beta1):
    density1 = (1 / (np.sqrt(2 * np.pi) * sigma1)) * np.exp(-0.5 * ((beta1 - mu1) / sigma1)**2)
    density2 = (1 / (np.sqrt(2 * np.pi) * sigma2)) * np.exp(-0.5 * ((beta2 - mu2) / sigma2)**2)

    return binomial_logit_2d(beta1, beta2) * density1 * density2

true_val_2d, err_2d = integrate.dblquad(
    integrand_2d,
    -np.inf, np.inf,
    lambda b1: -np.inf,
    lambda b1: np.inf,
    epsabs=1e-12,
    epsrel=1e-12
)

print("True value (2D quad):", true_val_2d)
print("Integration error:", err_2d)

def monte_carlo_2d(n_draws, seed=123):
    np.random.seed(seed)

    beta1_draws = np.random.normal(mu1, sigma1, n_draws)
    beta2_draws = np.random.normal(mu2, sigma2, n_draws)

    values = expit(beta1_draws * X1 + beta2_draws * X2)
    estimate = np.mean(values)

    return estimate

mc_20_2d = monte_carlo_2d(20)
mc_400_2d = monte_carlo_2d(400)

print("MC 20 draws (2D):", mc_20_2d, "error:", mc_20_2d - true_val_2d)
print("MC 400 draws (2D):", mc_400_2d, "error:", mc_400_2d - true_val_2d)

def gauss_hermite_2d(k):
    nodes, weights = hermgauss(k)

    total = 0.0

    for i in range(k):
        for j in range(k):
            beta1 = np.sqrt(2) * sigma1 * nodes[i] + mu1
            beta2 = np.sqrt(2) * sigma2 * nodes[j] + mu2

            value = expit(beta1 * X1 + beta2 * X2)

            total += weights[i] * weights[j] * value

    approximation = total / np.pi
    return approximation

gh_4_2d = gauss_hermite_2d(4)
gh_12_2d = gauss_hermite_2d(12)

print("GH k=4 (2D):", gh_4_2d, "error:", gh_4_2d - true_val_2d)
print("GH k=12 (2D):", gh_12_2d, "error:", gh_12_2d - true_val_2d)

True value (2D quad): 0.7144838053940904
Integration error: 9.956289417680044e-13
MC 20 draws (2D): 0.6733711673165269 error: -0.04111263807756349
MC 400 draws (2D): 0.7038299738471793 error: -0.01065383154691113
GH k=4 (2D): 0.7143713789717683 error: -0.00011242642232212052
GH k=12 (2D): 0.7144838098576901 error: 4.463599712067889e-09


### Part II Q7

In [32]:

results_2d = pd.DataFrame({
    "Method": ["Monte Carlo", "Monte Carlo", "Gauss-Hermite", "Gauss-Hermite"],
    "Points/Draws": [20, 400, 16, 144],
    "Estimate": [mc_20_2d, mc_400_2d, gh_4_2d, gh_12_2d],
    "Error": [
        mc_20_2d - true_val_2d,
        mc_400_2d - true_val_2d,
        gh_4_2d - true_val_2d,
        gh_12_2d - true_val_2d
    ]
})


print("Table for 1D integral")
print(results_1d.round(10))
print()


print("Table for 2D integral")
print(results_2d.round(10))


Table for 1D integral
          Method  Points/Draws  Estimate         Error
0    Monte Carlo            20  0.565680  1.418647e-02
1    Monte Carlo           400  0.537245 -1.424846e-02
2  Gauss-Hermite             4  0.551314 -1.793959e-04
3  Gauss-Hermite            12  0.551493 -6.860000e-08

Table for 2D integral
          Method  Points/Draws  Estimate         Error
0    Monte Carlo            20  0.673371 -4.111264e-02
1    Monte Carlo           400  0.703830 -1.065383e-02
2  Gauss-Hermite            16  0.714371 -1.124264e-04
3  Gauss-Hermite           144  0.714484  4.500000e-09


## Part III

In [33]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

### Part III Q1

Upload Data

In [34]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("sim_data_new.csv")

Saving sim_data_new.csv to sim_data_new (2).csv


In [36]:

from scipy.special import logsumexp

df["bath_family"] = df["bath_j"] * df["family_size_i"]

N_ind = df["individual_id_i"].nunique()

J = df.groupby("individual_id_i").size().iloc[0]  # alternatives per person

df = df.sort_values(["individual_id_i", "apt_id_j"]).copy()

def make_matrix(col):
    return df[col].to_numpy().reshape(N_ind, J)

log_sqft    = make_matrix("logsqfeet_j")
bathrooms   = make_matrix("bath_j")
bath_family = make_matrix("bath_family")
outdoor     = make_matrix("outdoor_j")
choice      = make_matrix("y_ij")

def neg_loglike_vec(theta):
    beta0, beta1, beta2, beta3, gamma = theta

    V = beta0 + beta1*log_sqft + beta2*bathrooms + beta3*bath_family + gamma*outdoor

    # Add outside option
    V_with_outside = np.concatenate([np.zeros((N_ind, 1)), V], axis=1)

    # Denominator for Logit
    log_denom = logsumexp(V_with_outside, axis=1)


    V_chosen = (V * choice).sum(axis=1)

    ll = (V_chosen - log_denom).sum()
    return -ll

theta0_logit = np.zeros(5)

result_logit = minimize(neg_loglike_vec, theta0_logit,  method="BFGS")


Results

In [37]:
param_names = ["beta0", "beta1", "beta2", "beta3", "gamma"]


print("Log-likelihood:", -result_logit.fun)
print()

for name, val in zip(param_names, result_logit.x):
    print(f"{name}: {val:.6f}")


Log-likelihood: -24789.875609378076

beta0: 14.770049
beta1: -0.822237
beta2: 0.692191
beta3: 0.042950
gamma: 0.768995


### Part III Q2

In [41]:

k = 12
nodes, weights = hermgauss(k)
gauss_w = weights / np.sqrt(np.pi)

def neg_loglike_mixed_vec(theta):
    beta0, beta1, beta2, beta3, gamma_mean, log_sigma = theta
    sigma = np.exp(log_sigma)

    # Base utility
    base_util = beta0 + beta1*log_sqft + beta2*bathrooms + beta3*bath_family

    # GH nodes
    gamma_nodes = np.sqrt(2) * sigma * nodes + gamma_mean


    V_full = base_util[None, :, :] + gamma_nodes[:, None, None] * outdoor[None, :, :]

    # Add outside option
    outside = np.zeros((k, N_ind, 1))
    V_with_out = np.concatenate([outside, V_full], axis=2)

    # Denominator for mixed logit
    log_denom = logsumexp(V_with_out, axis=2)

    V_chosen = (V_full * choice[None, :, :]).sum(axis=2)

    p_cond = np.exp(V_chosen - log_denom)


    # Integrate
    prob_i = (gauss_w[:, None] * p_cond).sum(axis=0)

    prob_i = np.maximum(prob_i, 1e-300)
    ll = np.log(prob_i).sum()

    return -ll

theta0_mixed = np.zeros(6)

result_mixed = minimize(neg_loglike_mixed_vec, theta0_mixed, method="L-BFGS-B")
print(result_mixed)

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 23962.53134549633
        x: [ 1.612e+01 -2.110e-01  2.696e-01  7.383e-02  1.047e+00
             7.353e-01]
      nit: 36
      jac: [-3.638e-04 -1.175e-01 -7.640e-03 -4.002e-03 -5.639e-02
             2.547e-02]
     nfev: 287
     njev: 41
 hess_inv: <6x6 LbfgsInvHessProduct with dtype=float64>


In [42]:
param_names_mixed = ["beta0", "beta1", "beta2", "beta3", "gamma_mean", "log_sigma"]

for name, val in zip(param_names_mixed, result_mixed.x):
    print(f"{name}: {val:.6f}")

sigma_hat = np.exp(result_mixed.x[-1])
print("\nImplied sigma:", sigma_hat)

beta0: 16.120677
beta1: -0.210952
beta2: 0.269555
beta3: 0.073826
gamma_mean: 1.047270
log_sigma: 0.735305

Implied sigma: 2.0861175988416027


Part III Q3

In [46]:
#Q1
var_cov_logit = result_logit.hess_inv
se_logit = np.sqrt(np.diag(var_cov_logit))

#Q2
var_cov_mixed = result_mixed.hess_inv.todense()
se_mixed = np.sqrt(np.diag(var_cov_mixed))

#Results
print("Standard Errors (Q1):")
for name, s in zip(param_names, se_logit):
    print(f"{name}: {s:.6f}")
print("Standard Errors (Q2):")
for name, s in zip(param_names_mixed, se_mixed):
    print(f"{name}: {s:.6f}")


Standard Errors (Q1):
beta0: 7.509189
beta1: 0.302211
beta2: 12.226544
beta3: 4.394172
gamma: 1.011195
Standard Errors (Q2):
beta0: 547.652782
beta1: 0.353032
beta2: 5.662143
beta3: 2.113652
gamma_mean: 0.873989
log_sigma: 0.768839
